# 🚢 Domácí úkol (Session 2): Random Forest – Klasifikace přežití na Titanicu
**Dataset:** `titanic_data.csv` (1 309 cestujících, 14 atributů)  
**Cíl úlohy:** Konstrukce a optimalizace modelu náhodného lesa (`RandomForestClassifier`) pro binární klasifikaci přežití (`survived`), včetně čištění zbytečných sloupců, one-hot kódování kategorií, rozdělení 70:30, hyperparametrického ladění přes `GridSearchCV` s optimalizací na metriku **`precision`** a detailního vyhodnocení přes `classification_report`.

---
### 📋 Kroky zadání:
1. **Načtení dat:** Stáhněte a načtěte `titanic_data.csv` do proměnné pomocí knihovny Pandas.
2. **Import tříd:** Importujte potřebné nástroje z knihovny `scikit-learn`.
3. **Čištění dat:** Odstraňte zbytečné sloupce z datasetu (`passengerid`, `name`, `ticket`, `cabin`).
4. **Kategorická transformace:** Proveďte transformaci kategoriálních proměnných (`sex`, `embarked`, `title`).
5. **Train-Test Split:** Rozdělte dataset na trénovací a testovací sadu v poměru 70:30 (`random_state=42`).
6. **Instance modelu:** Inicializujte `RandomForestClassifier` se specifikovanými parametry `random_state` a `n_jobs`.
7. **Mřížka parametrů:** Definujte slovník `params` pro klíče `max_depth`, `min_samples_leaf` a `n_estimators` (2 až 4 hodnoty na klíč).
8. **GridSearchCV:** Vytvořte objekt `GridSearchCV` pro výběr nejlepší kombinace s optimalizovanou metrikou `scoring='precision'`.
9. **Trénování:** Natrénujte nejlepší model na trénovacích datech `X_train, y_train`.
10. **Predikce:** Proveďte predikci na testovacích datech `X_test`.
11. **Metriky:** Vygenerujte souhrnný report `classification_report` a analyzujte výsledky.


In [1]:
# Krok 1 & 2: Import knihoven a načtení dat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

# Načtení Titanic datasetu
try:
    df = pd.read_csv('titanic_data.csv')
except FileNotFoundError:
    df = pd.read_csv('data/titanic_data.csv')

print(f"Dataset úspěšně načten! Tvar: {df.shape}")
df.head()

### 🔍 Průzkum dat a struktura atributů
Zkontrolujeme datové typy a přítomnost chybějících hodnot před odstraněním zbytečných sloupců.

In [2]:
print("Informace o sloupcích:")
print(df.info())
print("\nChybějící hodnoty v jednotlivých sloupcích:")
print(df.isnull().sum())
print("\nRozdělení cílové proměnné 'survived':")
print(df['survived'].value_counts(normalize=True))

### 🧹 Krok 3 & 4: Odstranění zbytečných sloupců a transformace kategorií
* **Zbytečné sloupce:** `passengerid` (pouhý unikátní index), `name` (jméno osoby s vysokou kardinalitou), `ticket` (číslo lístku bez obecné sémantiky), `cabin` (přes 77 % chybějících hodnot).
* **Kategoriální proměnné:** `sex` (pohlaví), `embarked` (přístav nalodění C/Q/S), `title` (titul Mr/Mrs/Miss/Master/Rev/Dr).
Použijeme One-Hot kódování přes `pd.get_dummies(..., drop_first=True)`.

In [3]:
# Krok 3: Odstranění zbytečných sloupců
drop_cols = ['passengerid', 'name', 'ticket', 'cabin']
df_cleaned = df.drop(columns=drop_cols)

# Krok 4: Transformace kategoriálních proměnných
cat_cols = ['sex', 'embarked', 'title']
df_encoded = pd.get_dummies(df_cleaned, columns=cat_cols, drop_first=True)

print(f"Tvar po One-Hot kódování: {df_encoded.shape}")
print("Seznam výsledných příznaků:", df_encoded.columns.tolist())
df_encoded.head()

### ✂️ Krok 5: Rozdělení na trénovací a testovací sadu (70:30, `random_state=42`)

In [4]:
X = df_encoded.drop(columns=['survived'])
y = df_encoded['survived']

# Rozdělení 70:30 se stratifikací pro zachování poměru přeživších
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Velikost trénovací sady: {X_train.shape[0]} vzorků")
print(f"Velikost testovací sady:   {X_test.shape[0]} vzorků")

### 🌲 Krok 6, 7 & 8: Inicializace Random Forest a GridSearchCV
* **Model:** `RandomForestClassifier(random_state=42, n_jobs=-1)`
* **Mřížka hyperparametrů `params`:**
  * `n_estimators`: [50, 100, 150]
  * `max_depth`: [4, 6, 8, 12]
  * `min_samples_leaf`: [1, 2, 4]
* **Optimalizovaná metrika:** `scoring='precision'` (přesnost predikce přežití)

In [5]:
# Krok 6: Instance Random Forest
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1)

# Krok 7: Slovník hyperparametrů
params = {
    'max_depth': [4, 6, 8, 12],
    'min_samples_leaf': [1, 2, 4],
    'n_estimators': [50, 100, 150]
}

# Krok 8: GridSearchCV s metrikou precision a 5-násobnou křížovou validací
grid_search = GridSearchCV(
    estimator=rf_classifier,
    param_grid=params,
    scoring='precision',
    cv=5,
    n_jobs=-1
)

# Krok 9: Trénování modelu na trénovacích datech
grid_search.fit(X_train, y_train)

print("Nejlepší nalezená sada hyperparametrů:", grid_search.best_params_)
print(f"Nejlepší průměrná Precision v křížové validaci: {grid_search.best_score_:.4f}")

### 🎯 Krok 10 & 11: Predikce na testovací sadě a `classification_report`
Vyhodnotíme finální natrénovaný model `best_estimator_` na nezávislé testovací sadě.

In [6]:
# Krok 10: Predikce pomocí nejlepšího modelu
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

# Krok 11: Výpis souhrnného reportu metrik
print("=== CLASSIFICATION REPORT (Testovací sada 393 vzorků) ===\n")
print(classification_report(y_test, y_pred, target_names=['Nepřežil (0)', 'Přežil (1)']))

# Konfúzní matice
cm = confusion_matrix(y_test, y_pred)
print("Konfúzní matice:\n", cm)

### 📊 Doplňková vizualizace: Důležitost příznaků (Feature Importance)
Které faktory měly na přežití pasažérů Titanicu největší vliv?

In [7]:
importances = best_rf.feature_importances_
feat_imp_df = pd.DataFrame({
    'Příznak': X.columns,
    'Důležitost': importances
}).sort_values('Důležitost', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feat_imp_df['Příznak'], feat_imp_df['Důležitost'], color='#2563eb')
plt.title('Důležitost příznaků (Random Forest Feature Importance) – Titanic')
plt.xlabel('Gini Importance')
plt.tight_layout()
plt.show()